<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/04_registration/register_inhale_exhale_4dct.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Register inhale and exhale phases of a lung 4D-CT using ANTs

## Introduction

This notebook performs deformable registration between the **inhale** and **exhale** phases of a lung 4D-CT using ANTsPy through pyCERR's `cerr.registration.ants_reg` module. The recovered deformation is the basis for CT-ventilation biomarkers.

The exhale phase is the **fixed** image and the inhale phase is the **moving** image. The similarity metric is restricted to a **lung mask** (union of left and right lungs, slightly expanded) on each phase so the registration is driven by lung motion rather than the chest wall.

## I/O

* **Inputs**: DICOM directories for the exhale and inhale phases, and a lung segmentation (NIfTI) for each phase (e.g. from an AI model).
* **Output**: a pyCERR `planC` with the warped inhale CT added and the ANTs transform written to disk.

All paths below are **placeholders** — edit them to point at your data.

## License

By downloading the software you are agreeing to the following terms and conditions as well as to the Terms of Use of CERR software.

**`THE SOFTWARE IS PROVIDED "AS IS" AND CERR DEVELOPMENT TEAM AND ITS COLLABORATORS DO NOT MAKE ANY WARRANTY, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE, NOR DO THEY ASSUME ANY LIABILITY OR RESPONSIBILITY FOR THE USE OF THIS SOFTWARE.`**

`This software is for research purposes only and has not been approved for clinical use.`

## Install pyCERR

In [ ]:
# Install pyCERR with the ANTs registration extra (pulls in antspyx).
!pip install "pyCERR[ants] @ git+https://github.com/cerr/pyCERR.git@testing"

## Imports

In [ ]:
import os
import numpy as np
from cerr import plan_container as pc
from cerr.registration import ants_reg
from cerr.contour import rasterseg as rs
from cerr.dataclasses import structure as cerrStr

## Define input/output paths

Replace the placeholder strings with paths to your own data.

In [ ]:
# --- Placeholders: edit these ---
exhaleDicomDir   = 'path/to/exhale_ct_dicom_dir'   # fixed (base)
inhaleDicomDir   = 'path/to/inhale_ct_dicom_dir'   # moving
exhaleLungSegNii = 'path/to/exhale_lung_seg.nii.gz'
inhaleLungSegNii = 'path/to/inhale_lung_seg.nii.gz'
transformSaveDir = 'path/to/output/transforms'

os.makedirs(transformSaveDir, exist_ok=True)

## Load the scans and lung segmentations

The lung segmentation NIfTI is assumed to label the left and right lungs as `1` and `2`. Adjust the `labels_dict` to match your segmentation.

In [ ]:
planC = pc.loadDcmDir(exhaleDicomDir)            # scan 0 = exhale (fixed)
planC = pc.loadDcmDir(inhaleDicomDir, {}, planC) # scan 1 = inhale (moving)

baseScanNum = 0
movScanNum  = 1

lungLabels = {'Lung_L': 1, 'Lung_R': 2}
planC = pc.loadNiiStructure(exhaleLungSegNii, baseScanNum, planC, lungLabels)
planC = pc.loadNiiStructure(inhaleLungSegNii, movScanNum, planC, lungLabels)

for i, st in enumerate(planC.structure):
    print(i, st.structureName)

In [ ]:
# Set these after inspecting the printout above: the lung structures on
# the fixed (exhale) and moving (inhale) scans.
baseLungStrNums = [0, 1]
movLungStrNums  = [2, 3]

## Build lung masks

Union the left and right lung masks on each phase, then expand the surface slightly so the metric region comfortably contains the moving lung.

In [ ]:
baseMask3M = np.zeros(planC.scan[baseScanNum].getScanSize(), dtype=bool)
for strNum in baseLungStrNums:
    baseMask3M |= rs.getStrMask(strNum, planC)

movMask3M = np.zeros(planC.scan[movScanNum].getScanSize(), dtype=bool)
for strNum in movLungStrNums:
    movMask3M |= rs.getStrMask(strNum, planC)

# Optional: expand the mask surface by a small margin (cm).
expansionMargin = 0.1
planC = pc.importStructureMask(baseMask3M, baseScanNum, 'Lungs_union', planC)
planC = cerrStr.getSurfaceExpand(len(planC.structure) - 1, expansionMargin, planC)
baseMask3M = rs.getStrMask(len(planC.structure) - 1, planC)

planC = pc.importStructureMask(movMask3M, movScanNum, 'Lungs_union', planC)
planC = cerrStr.getSurfaceExpand(len(planC.structure) - 1, expansionMargin, planC)
movMask3M = rs.getStrMask(len(planC.structure) - 1, planC)

## Register

Here we use `'antsRegistrationSyN[bo]'` (b-spline SyN, `o`utput warped image) with an identity initial transform since the two phases already share a frame of reference. The masks are applied at every stage.

In [ ]:
planC = ants_reg.registerScansAnts(
    planC, baseScanNum, planC, movScanNum,
    transformSaveDir=transformSaveDir,
    typeOfTransform='antsRegistrationSyN[bo]',
    initialTransform='identity',
    baseMask3M=baseMask3M, movMask3M=movMask3M, maskAllStages=True)

deformS = planC.deform[-1]
print('Transform written to:', deformS.deformOutFilePath)

## Visualize the result

In [ ]:
# Quick visual check: central axial slice of fixed, moving, and
# warped-moving scans. The warped-moving scan is the last one added.
import numpy as np
import matplotlib.pyplot as plt

fixed3M = planC.scan[baseScanNum].getScanArray()
moving3M = planC.scan[movScanNum].getScanArray()
warped3M = planC.scan[-1].getScanArray()

slc = fixed3M.shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(fixed3M[:, :, slc], cmap='gray'); ax[0].set_title('Fixed')
ax[1].imshow(moving3M[:, :, min(slc, moving3M.shape[2] - 1)], cmap='gray')
ax[1].set_title('Moving (before)')
ax[2].imshow(warped3M[:, :, slc], cmap='gray')
ax[2].set_title('Warped moving')
for a in ax:
    a.axis('off')
plt.tight_layout(); plt.show()